### Trasformation 

\begin{equation}
E: (a, e, M, w)  \rightarrow X: (x,y,v_x, v_y)
\end{equation}

\begin{equation}
    \mathbb{J}_{X o E} =
    \begin{pmatrix} 
    \partial_ax & \partial_ex & \partial_wx & \partial_Mx\\
    \partial_ay & \partial_ey & \partial_wy & \partial_My \\
    \partial_av_x & \partial_ev_x & \partial_wv_x & \partial_Mv_x \\
    \partial_av_y & \partial_ev_y & \partial_wv_y & \partial_Mv_y 
    \end{pmatrix}
\end{equation}

In [63]:
import numpy as np
from Utils import CanonicalUnits, OrbitalElements, JaccobianComponents, Kepler, GravitationalParameters, X2E, computeNumericalJacobian
import spiceypy as spy
import scipy.optimize as optimize
from tqdm import tqdm

In [ ]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [20]:
xp2ep([1,2,1,1], mu)

array([-0.9045085 ,  1.4510592 ,  3.21441268,  1.63560027])

In [21]:
def xp2ep(X: np.array, mu: float) -> np.array:
    E = X2E(np.array([X[0], X[1], 0, X[2], X[3], 0]), mu)
    a = E[0]/(1-E[1])
    e = E[1]
    w = E[4]
    M = E[5]

    E = np.array([a, e, w, M])
    return E

def jacobian_EoX_num4(q: float, e: float, w: float, M: float, mu: float) -> np.array:
    E = [q, e, 0, 0, w, M]
    X = spy.conics(E+[0, mu], 0)
    X = np.array([X[0], X[1], X[3], X[4]])
    dX=np.array([1e-3]*4)
    args=dict(mu=mu)
    E_num, JEoX=computeNumericalJacobian(xp2ep,X,dX,**args)
    return JEoX

    

def jacobian_EoX_num(q: float, e: float, w: float, M: float, mu: float) -> np.array:
    E = [q, e, 0, 0, w, M]
    X = spy.conics(E+[0, mu], 0)
    dX=np.array([1e-3]*6)
    args=dict(mu=mu)
    E_num, JEoX=computeNumericalJacobian(X2E,X,dX,**args)

    partial_xq = JEoX[0,0]
    partial_xe = JEoX[1,0]
    partial_xw = JEoX[4,0]
    partial_xM = JEoX[5,0]

    partial_yq = JEoX[0,1]
    partial_ye = JEoX[1,1]
    partial_yw = JEoX[4,1]
    partial_yM = JEoX[5,1]

    partial_vxq = JEoX[0,3]
    partial_vxe = JEoX[1,3]
    partial_vxw = JEoX[4,3]
    partial_vxM = JEoX[5,3]

    partial_vyq = JEoX[0,4]
    partial_vye = JEoX[1,4]
    partial_vyw = JEoX[4,4]
    partial_vyM = JEoX[5,4]

    Matrix = np.array([[partial_xq, partial_yq, partial_vxq, partial_vyq], 
                        [partial_xe, partial_ye, partial_vxe, partial_vye], 
                        [partial_xw, partial_yw, partial_vxw, partial_vyw], 
                        [partial_xM, partial_yM, partial_vxM, partial_vyM]])
    return Matrix


In [22]:
q = 1
e = 0.5
w = 0.3
M = 0.5
mu = 1
jacobian_EoX_num4(q, e, w, M, mu) 

array([[-4.57249208e-01,  4.24447456e+00, -7.45969498e+00,
         2.42049928e+00],
       [-5.42027500e-01,  4.49666828e-01, -1.13038689e+00,
         1.08725855e+00],
       [ 7.63102227e-01,  1.52446327e+00, -4.52152791e+00,
        -9.58883052e-01],
       [-1.52790202e-03, -1.53416667e+00,  4.12311233e+00,
        -1.09440147e+00]])

In [23]:
jacobian_EoX_num(q, e, w, M, mu) 

array([[ 8.55431332e-01,  1.22289825e+00, -1.46899661e+00,
        -9.64275208e-01],
       [-5.42027500e-01,  4.49666828e-01, -1.13038689e+00,
         1.08725855e+00],
       [ 7.63102227e-01,  1.52446327e+00, -4.52152791e+00,
        -9.58883052e-01],
       [-1.52790202e-03, -1.53416667e+00,  4.12311233e+00,
        -1.09440147e+00]])

In [67]:
def trasformation_aewE_to_xyvxvy(a: float, e: float, w: float, M: float) -> tuple[float, float, float, float]:
    Omega = 0
    i = 0
    mu = CanonicalUnits().mu
    q = a*(1-e)

    state_vec = spy.conics([q, e, i, w, Omega, M]+[0, mu], 0)
    x = state_vec[0]
    y = state_vec[1]
    vx = state_vec[3]
    vy = state_vec[4]

    return x, y, vx, vy

def trasformation_xyvxvy_to_aewE(x: float, y: float, vx: float, vy: float) -> tuple[float, float, float, float]:

    mu = CanonicalUnits().mu
    elements = spy.oscelt([x, y, 0, vx, vy, 0], et=0, mu=mu)
    q = elements[0]
    e = elements[1]
    w = elements[4]
    M = elements[5]
    a = q/(1-e)

    return a, e, w, M

In [68]:
N = int(1e5)

a_uniform = np.random.uniform(0, 2, N)
e_uniform = np.random.uniform(0, 1, N)
w_uniform = np.random.uniform(0, 2*np.pi, N)
M_uniform = np.random.uniform(0, 2*np.pi, N)
q_uniform = a_uniform*(1-e_uniform)

x_uniform, y_uniform, vx_uniform, vy_uniform = trasformation_aewE_to_xyvxvy(a_uniform[0], e_uniform[0], w_uniform[0], M_uniform[0])
x_uniform, y_uniform, vx_uniform, vy_uniform

(0.9603065909406825,
 0.5733474652281444,
 -3.0610154876468525,
 4.966857560437392)

In [71]:
xyvxvy = np.zeros((N, 4))

for el in tqdm(range(N)):
    a = a_uniform[el]
    e = e_uniform[el]
    w = w_uniform[el]
    M = M_uniform[el]

    x, y, vx, vy = trasformation_aewE_to_xyvxvy(a, e, w, M)
    xyvxvy[el] = np.array([x, y, vx, vy])

100%|██████████| 100000/100000 [00:00<00:00, 103290.38it/s]


In [72]:
xyvxvy

array([[ 0.96030659,  0.57334747, -3.06101549,  4.96685756],
       [ 2.45943175, -1.82702678,  0.8697108 ,  1.58803628],
       [-0.56018813, -0.64362733,  4.24877835,  1.90313095],
       ...,
       [ 0.12634584, -0.70049276,  6.49268569, -1.24445404],
       [-1.48120899,  0.68755929, -1.82488926, -4.11854058],
       [-1.03403418,  1.27517441, -1.4853383 ,  1.36171772]])

In [73]:
def P_aewM() -> float:
    a_max = 2; a_min = 0
    e_max = 1; e_min = 0
    w_max = 2*np.pi; w_min = 0
    M_max = 2*np.pi; M_min = 0
    return 1/(a_max - a_min) * 1/(e_max - e_min) * 1/(w_max - w_min) * 1/(M_max - M_min)

def P_xyvxvy(x: float, y: float, vx: float, vy: float) -> float:
    a, e, w, M = trasformation_xyvxvy_to_aewE(x, y, vx, vy)
    q = a*(1-e)
    J = jacobian_EoX_num(q, e, w, M, mu)
    det = np.linalg.det(J)/(1-e)
    #det = 1.0/np.linalg.det(J)
    P = P_aewM() * abs(det)
    return P

def P_xyvxvy_vectorized(x: np.array, y: np.array, vx: np.array, vy: np.array) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    vx = np.asarray(vx)
    vy = np.asarray(vy)

    # Prepare output array
    shape = np.broadcast(x, y, vx, vy).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()

    for idx in range(x_flat.size):
        a, e, w, M = trasformation_xyvxvy_to_aewE(x_flat[idx], y_flat[idx], vx_flat[idx], vy_flat[idx])
        q = a*(1-e)
        J = jacobian_EoX_num(q, e, w, M, mu)
        det = np.linalg.det(J)/(1-e)
        #det = 1.0/np.linalg.det(J)
        P.flat[idx] = P_aewM() * abs(det)

    return P.reshape(shape)


## Probabilidad en un hipercubo